# DX 704 Week 7 Project

This week's project will investigate issues in a quadcopter controller based using a linear quadratic regulator.
You will start with an existing model of the system and logs from a quadcopter based on it, investigate discrepancies, and ultimately train a new model of the system dynamics.

The full project description and a template notebook are available on GitHub: [Project 7 Materials](https://github.com/bu-cds-dx704/dx704-project-07).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Introduction

You've just joined a drone startup.
On your first day, you join your new team to watch a test flight for a new quadcopter prototype.
Watching the prototype fly, the team comments that it is not as smooth as usual and suspects that something is off in the controller.
They provide you a copy of the dynamics model and log data from the prototype to investigate.

The quadcopter control model is a slightly more sophisticated version of the 1D quadcopter problem previously considered.

The state vector $\mathbf{x}_t$ now includes an acceleration component, and the action vector now has a servo control for the throttle instead of a raw force component.
\begin{array}{rcl}
\mathbf{x}_t & = & \begin{bmatrix} x_t \\ v_t \\ a_t \end{bmatrix} \\
\mathbf{u_t} & = & \begin{bmatrix} u_t \end{bmatrix}
\end{array}

## Part 1: Reconstruct the Control Matrix

You are provided the dynamics model in the files `model-A.tsv`, `model-B.tsv`, `cost-Q.tsv` and `cost-R.tsv`.
Recompute the control matrix $\mathbf{K}$ to be used in the infinite horizon linear quadratic regulator problem.

In [11]:
# YOUR CHANGES HERE

import numpy as np
import pandas as pd
from scipy.linalg import solve_discrete_are

# Helper to load TSV matrices with row/column labels
def load_tsv_matrix(path):
    df = pd.read_csv(path, sep="\t")
    return df.set_index(df.columns[0]).astype(float).values

# Load system and cost matrices
A = load_tsv_matrix("model-A.tsv")
B = load_tsv_matrix("model-B.tsv")
Q = load_tsv_matrix("cost-Q.tsv")
R = load_tsv_matrix("cost-R.tsv")

# Solve the discrete-time algebraic Riccati equation
P = solve_discrete_are(A, B, Q, R)

# Infinite-horizon discrete LQR gain
K = np.linalg.inv(R + B.T @ P @ B) @ (B.T @ P @ A)

print("A =\n", A)
print("\nB =\n", B)
print("\nQ =\n", Q)
print("\nR =\n", R)
print("\nP =\n", P)
print("\nK =\n", K)


A =
 [[1. 1. 0.]
 [0. 1. 1.]
 [0. 0. 1.]]

B =
 [[0.]
 [0.]
 [1.]]

Q =
 [[5. 0. 0.]
 [0. 1. 0.]
 [0. 0. 2.]]

R =
 [[5.]]

P =
 [[19.50090752 27.77808565 14.94947739]
 [27.77808565 65.61201283 43.35619783]
 [14.94947739 43.35619783 39.69737486]]

K =
 [[0.33445985 1.30445413 1.85813088]]


Save $\mathbf{K}$ in a file "control-K-intended.tsv" with columns x, v and a.

In [12]:
# YOUR CHANGES HERE
df_K = pd.DataFrame(K, columns=["x", "v", "a"])

# Save to TSV file
output_path = "control-K-intended.tsv"
df_K.to_csv(output_path, sep="\t", index=False)

print("Saved control matrix to:", output_path)
print(df_K)

Saved control matrix to: control-K-intended.tsv
         x         v         a
0  0.33446  1.304454  1.858131


Submit "control-K-intended.tsv" in Gradescope.

## Part 2: Recompute the Actions for the Logged States

You get access to the log data for the prototype as the file "qc-log.tsv".
It conveniently saves all the state and actions made.
Recompute the actions based on your $\mathbf{K}$ matrix computed in part 1.

In [13]:
# YOUR CHANGES HERE

import pandas as pd
import numpy as np

# Load log data
log_df = pd.read_csv("qc-log.tsv", sep="\t")

# Load K from file created earlier
K_df = pd.read_csv("control-K-intended.tsv", sep="\t")
K = K_df.values.flatten()

# Extract state matrix (x, v, a)
X = log_df[["x", "v", "a"]].values

# Recompute actions using LQR control law
u_recomputed = -X @ K

# Add recomputed actions to dataframe
log_df["u_recomputed"] = u_recomputed

# Save results
log_df.to_csv("qc-log-recomputed.tsv", sep="\t", index=False)

print(log_df.head())


   index         x         v         a         u  u_recomputed
0      0 -5.000000  0.000000  0.000000  1.702188      1.672299
1      1 -5.000000 -0.017022  1.531969 -1.263200     -1.152095
2      2 -5.018724  1.452683  0.548285 -1.249321     -1.235183
3      3 -3.420773  1.840779 -0.521275 -0.212127     -0.288504
4      4 -1.395916  1.163611 -0.764317  0.452895      0.369202


Save your computed actions as "qc-check.tsv" with columns "index" and "u_check".

In [14]:
# YOUR CHANGES HERE

import pandas as pd
import numpy as np

# Load log data
log_df = pd.read_csv("qc-log.tsv", sep="\t")

# Load K
K = pd.read_csv("control-K-intended.tsv", sep="\t").values.flatten()

# Extract states
X = log_df[["x", "v", "a"]].values

# Compute LQR action: u = -Kx
u_check = -X @ K

# Create output dataframe
df_check = pd.DataFrame({
    "index": log_df.index,
    "u_check": u_check
})

# Save to TSV
df_check.to_csv("qc-check.tsv", sep="\t", index=False)

print(df_check.head())


   index   u_check
0      0  1.672299
1      1 -1.152095
2      2 -1.235183
3      3 -0.288504
4      4  0.369202


Submit "qc-check.tsv" in Gradescope.

## Part 3: Reverse Engineer the Actual Control Matrix

Now that you have found a systematic difference between your computed actions and the logged actions, estimate $
$, the control matrix that was actually used to choose actions in the prototype.

Hint: With a linear quadratic regulator, the optimal actions are always linear combinations of the state that are calculated using the control matrix.
You can use linear regression to reverse-engineer the coefficients in the control matrix.

In [15]:
# YOUR CHANGES HERE

import pandas as pd
import numpy as np

# Load log data
log_df = pd.read_csv("qc-log.tsv", sep="\t")

# Build regression inputs
X = log_df[["x", "v", "a"]].values
u = log_df["u"].values

# Solve X @ k = -u  for k
K_actual, residuals, rank, s = np.linalg.lstsq(X, -u, rcond=None)

print("Estimated actual control matrix K:")
print(K_actual.reshape(1, -1))

print("\nResiduals:", residuals)
print("Rank:", rank)


Estimated actual control matrix K:
[[0.34043755 1.30012023 1.95011696]]

Residuals: [1.41189339e-31]
Rank: 3


Save $\mathbf{K}_{\mathrm{actual}}$ in "control-K-actual.tsv" with the same format as "control-K-intended.tsv".

In [16]:
# YOUR CHANGES HERE

import pandas as pd
import numpy as np

# Load log data
log_df = pd.read_csv("qc-log.tsv", sep="\t")

# Extract states and actions
X = log_df[["x", "v", "a"]].values
u = log_df["u"].values

# Estimate K_actual using least squares: X @ K = -u
K_actual, _, _, _ = np.linalg.lstsq(X, -u, rcond=None)

# Save in same format as control-K-intended.tsv
df_K_actual = pd.DataFrame([K_actual], columns=["x", "v", "a"])

df_K_actual.to_csv("control-K-actual.tsv", sep="\t", index=False)

print("Saved control matrix to control-K-actual.tsv")
print(df_K_actual)


Saved control matrix to control-K-actual.tsv
          x        v         a
0  0.340438  1.30012  1.950117


Submit "control-k-actual.tsv" in Gradescope.

## Part 4: Recompute the System Dynamics from the Log Data

On further investigation, it turns out that the control matrix $\mathbf{K}$ in the prototype was modified to compensate for state prediction errors.
You would like to recompute the $\mathbf{A}$ and $\mathbf{B}$ matrices using the log data since they are used to predict the next states.
However, since the action vector $\mathbf{u}_t$ is linearly dependent on the state via $\mathbf{u}_t=-\mathbf{K} \mathbf{x}_t$, you need a new data set so you can separate the effects of the $\mathbf{A}$ and $\mathbf{B}$ matrices.
Your co-workers kindly provide a new file "qc-train.tsv" where noise was added to each action.
Estimate the true $\mathbf{A}$ and $\mathbf{B}$ matrices based on this file.

In [19]:
import pandas as pd
import numpy as np

# Load training data
df = pd.read_csv("qc-train.tsv", sep="\t")

# Current state and action at time t
X_t = df[["x", "v", "a"]].iloc[:-1].values
U_t = df[["u"]].iloc[:-1].values

# Next state at time t+1 from the next row
X_next = df[["x", "v", "a"]].iloc[1:].values

# Combine state and action into one design matrix
Z = np.hstack([X_t, U_t])   # columns: x, v, a, u

# Solve Z @ Theta = X_next
Theta, residuals, rank, s = np.linalg.lstsq(Z, X_next, rcond=None)

# Extract A and B
A_est = Theta[:3, :].T      # 3x3
B_est = Theta[3:, :].T      # 3x1

print("Estimated A matrix:")
print(A_est)

print("\nEstimated B matrix:")
print(B_est)

print("\nResiduals:")
print(residuals)


Estimated A matrix:
[[ 1.00000000e+00  1.10000000e+00  2.88657986e-15]
 [-1.28195498e-16  9.00000000e-01  9.50000000e-01]
 [-9.31378031e-17  5.55111512e-16  1.10000000e+00]]

Estimated B matrix:
[[-1.11022302e-16]
 [-1.00000000e-02]
 [ 9.00000000e-01]]

Residuals:
[1.04885193e-29 1.44885838e-30 1.97050945e-30]


Save $\mathbf{A}$ and $\mathbf{B}$ to "model-A-new.tsv" and "model-B-new.tsv" respectively.

In [20]:
# YOUR CHANGES HERE
import pandas as pd
import numpy as np

# Load training data
df = pd.read_csv("qc-train.tsv", sep="\t")

# Current state and action at time t
X_t = df[["x", "v", "a"]].iloc[:-1].values
U_t = df[["u"]].iloc[:-1].values

# Next state at time t+1
X_next = df[["x", "v", "a"]].iloc[1:].values

# Build regression matrix
Z = np.hstack([X_t, U_t])   # columns: x, v, a, u

# Solve Z @ Theta = X_next
Theta, _, _, _ = np.linalg.lstsq(Z, X_next, rcond=None)

# Extract A and B
A_est = Theta[:3, :].T
B_est = Theta[3:, :].T

# Save A
A_df = pd.DataFrame(A_est, columns=["x", "v", "a"], index=["x", "v", "a"])
A_df.to_csv("model-A-new.tsv", sep="\t")

# Save B
B_df = pd.DataFrame(B_est, columns=["u"], index=["x", "v", "a"])
B_df.to_csv("model-B-new.tsv", sep="\t")

print("Saved model-A-new.tsv and model-B-new.tsv")
print("\nA matrix:")
print(A_df)
print("\nB matrix:")
print(B_df)


Saved model-A-new.tsv and model-B-new.tsv

A matrix:
              x             v             a
x  1.000000e+00  1.100000e+00  2.886580e-15
v -1.281955e-16  9.000000e-01  9.500000e-01
a -9.313780e-17  5.551115e-16  1.100000e+00

B matrix:
              u
x -1.110223e-16
v -1.000000e-02
a  9.000000e-01


Submit "model-A-new.tsv" and "model-B-new.tsv" in Gradescope.

## Part 5: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 6: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.